# Book Description Emotion Analysis

This notebook performs emotion analysis on book descriptions using a pre-trained transformer model. We'll:

1. **Load the book dataset** with genre classifications
2. **Set up an emotion classification model** to detect emotions in text
3. **Analyze emotions** at the sentence level for more granular insights
4. **Extract maximum emotion scores** for each book across all sentences
5. **Merge emotion scores** back into the main dataset
6. **Export the enriched dataset** with emotion features


In [13]:
# Import required libraries
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm

# Load the book dataset with genre classifications
df_books = pd.read_csv("books_with_categories.csv")

print(f"Dataset loaded: {len(df_books)} books")
print(f"Columns: {list(df_books.columns)}")
df_books.head()


Dataset loaded: 5197 books
Columns: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'full_title', 'indexed_content', 'genre']


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,full_title,indexed_content,genre
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction


## 1. Setting Up Emotion Classification Model

We'll use a pre-trained emotion classification model that can detect seven different emotions in text. This model analyzes the emotional tone of text, which can be useful for understanding the mood and atmosphere of book descriptions.


In [14]:
# Initialize the emotion classification pipeline
# Using DistilRoBERTa-based model fine-tuned for emotion detection
emotion_analyzer = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,  # Return all emotion scores, not just the top one
    device="mps"  # Use Metal Performance Shaders on Mac
)

# Test the model with a simple example
test_result = emotion_analyzer("I love this!")
print("Emotion classification model initialized successfully")
print(f"\nTest result: {test_result}")


/opt/anaconda3/envs/recommender/lib/python3.11/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Emotion classification model initialized successfully

Test result: [[{'label': 'joy', 'score': 0.9771687984466553}, {'label': 'surprise', 'score': 0.00852868054062128}, {'label': 'neutral', 'score': 0.00576459476724267}, {'label': 'anger', 'score': 0.004419783595949411}, {'label': 'sadness', 'score': 0.002092391485348344}, {'label': 'disgust', 'score': 0.001611992483958602}, {'label': 'fear', 'score': 0.00041385198710486293}]]


## 2. Testing on Book Descriptions

Let's test the emotion analyzer on a sample book description to see how it works. We'll analyze both the full description and individual sentences to understand the difference.


In [15]:
# Get a sample book description
sample_description = df_books['description'].iloc[0]

print("Sample book description:")
print(sample_description[:300] + "...")
print("\n" + "="*60)


Sample book description:
A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of t...



In [16]:
# Analyze the full description as a single text
full_text_emotions = emotion_analyzer(sample_description)

print("Emotion scores for full description:")
print(f"Dominant emotion: {full_text_emotions[0][0]['label']} (score: {full_text_emotions[0][0]['score']:.4f})")
print("\nAll emotion scores:")
for emotion in full_text_emotions[0]:
    print(f"  {emotion['label']}: {emotion['score']:.4f}")


Emotion scores for full description:
Dominant emotion: fear (score: 0.6548)

All emotion scores:
  fear: 0.6548
  neutral: 0.1699
  sadness: 0.1164
  surprise: 0.0207
  disgust: 0.0191
  joy: 0.0152
  anger: 0.0039


## 3. Sentence-Level Analysis

Analyzing at the sentence level provides more granular emotion detection. Different sentences in a description may convey different emotions, and we want to capture the maximum emotional intensity across all sentences.


In [17]:
# Split description into sentences and analyze each
description_sentences = sample_description.split(".")
# Filter out empty sentences
description_sentences = [s.strip() for s in description_sentences if s.strip()]

print(f"Number of sentences: {len(description_sentences)}")
print("\nFirst few sentences:")
for i, sentence in enumerate(description_sentences[:3]):
    print(f"\nSentence {i+1}: {sentence[:100]}...")


Number of sentences: 7

First few sentences:

Sentence 1: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an asto...

Sentence 2: John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of prea...

Sentence 3: It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in record...


In [18]:
# Analyze emotions for each sentence
sentence_emotions = emotion_analyzer(description_sentences)

print("Emotion analysis for each sentence:")
for idx, sentence_result in enumerate(sentence_emotions[:3]):
    top_emotion = sentence_result[0]
    print(f"\nSentence {idx+1}:")
    print(f"  Top emotion: {top_emotion['label']} (score: {top_emotion['score']:.4f})")
    print(f"  Sentence preview: {description_sentences[idx][:80]}...")


Emotion analysis for each sentence:

Sentence 1:
  Top emotion: surprise (score: 0.7296)
  Sentence preview: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decad...

Sentence 2:
  Top emotion: neutral (score: 0.4663)
  Sentence preview: John Ames is a preacher, the son of a preacher and the grandson (both maternal a...

Sentence 3:
  Top emotion: neutral (score: 0.6978)
  Sentence preview: It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he i...


## 4. Creating Emotion Score Extraction Function

We'll create a function that extracts the maximum emotion score for each emotion type across all sentences in a description. This gives us a single score per emotion per book, representing the peak emotional intensity.


In [19]:
# Define the emotion labels we're tracking
emotion_types = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

def extract_peak_emotion_scores(emotion_predictions):
    """
    Extract the maximum (peak) emotion score for each emotion type
    across multiple sentence predictions.
    
    Args:
        emotion_predictions: List of emotion prediction results for each sentence
        
    Returns:
        Dictionary with maximum score for each emotion type
    """
    # Initialize dictionary to store scores for each emotion
    emotion_score_arrays = {emotion: [] for emotion in emotion_types}
    
    # Process each sentence's predictions
    for sentence_prediction in emotion_predictions:
        # Sort predictions by label to ensure consistent ordering
        sorted_emotions = sorted(sentence_prediction, key=lambda x: x["label"])
        
        # Extract score for each emotion type
        for emotion_idx, emotion_type in enumerate(emotion_types):
            emotion_score_arrays[emotion_type].append(sorted_emotions[emotion_idx]["score"])
    
    # Return maximum score for each emotion across all sentences
    return {emotion: np.max(scores) for emotion, scores in emotion_score_arrays.items()}

# Test the function
test_scores = extract_peak_emotion_scores(sentence_emotions)
print("Peak emotion scores for sample book:")
for emotion, score in test_scores.items():
    print(f"  {emotion}: {score:.4f}")


Peak emotion scores for sample book:
  anger: 0.0296
  disgust: 0.3382
  fear: 0.9840
  joy: 0.9490
  sadness: 0.6978
  surprise: 0.9561
  neutral: 0.7296


## 5. Processing All Books

Now we'll process all books in the dataset to extract emotion scores. This will take some time as we analyze each book's description sentence by sentence.


In [20]:
# Initialize data structures for storing results
book_isbns = []
emotion_data = {emotion: [] for emotion in emotion_types}

print(f"Processing {len(df_books)} books for emotion analysis...")
print("This may take several minutes...\n")

# Process each book
for book_idx in tqdm(range(len(df_books)), desc="Analyzing emotions"):
    # Get book ISBN and description
    book_isbn = df_books['isbn13'].iloc[book_idx]
    book_description = df_books['description'].iloc[book_idx]
    
    # Split description into sentences
    sentences = [s.strip() for s in book_description.split(".") if s.strip()]
    
    # Skip if no valid sentences
    if not sentences:
        # Fill with zeros if no description
        book_isbns.append(book_isbn)
        for emotion in emotion_types:
            emotion_data[emotion].append(0.0)
        continue
    
    # Analyze emotions for all sentences
    emotion_predictions = emotion_analyzer(sentences)
    
    # Extract peak emotion scores
    peak_scores = extract_peak_emotion_scores(emotion_predictions)
    
    # Store results
    book_isbns.append(book_isbn)
    for emotion in emotion_types:
        emotion_data[emotion].append(peak_scores[emotion])


Processing 5197 books for emotion analysis...
This may take several minutes...



Analyzing emotions: 100%|██████████| 5197/5197 [05:10<00:00, 16.72it/s]


## 6. Creating Emotion Scores DataFrame

We'll create a DataFrame with all the emotion scores and merge it with the original book data.


In [21]:
# Create DataFrame from emotion scores
emotion_scores_df = pd.DataFrame(emotion_data)
emotion_scores_df['isbn13'] = book_isbns

print(f"Emotion scores DataFrame created: {len(emotion_scores_df)} books")
print(f"\nEmotion score statistics:")
print(emotion_scores_df[emotion_types].describe())

emotion_scores_df.head()


Emotion scores DataFrame created: 5197 books

Emotion score statistics:
             anger      disgust         fear          joy      sadness  \
count  5197.000000  5197.000000  5197.000000  5197.000000  5197.000000   
mean      0.140612     0.165510     0.284131     0.273391     0.686917   
std       0.231045     0.240864     0.348272     0.324584     0.317771   
min       0.000606     0.000416     0.000296     0.000398     0.000728   
25%       0.010625     0.014694     0.014688     0.017454     0.510199   
50%       0.032882     0.049859     0.085159     0.095303     0.847774   
75%       0.140976     0.202500     0.525637     0.498320     0.929939   
max       0.989582     0.989417     0.995216     0.993873     0.973930   

          surprise      neutral  
count  5197.000000  5197.000000  
mean      0.172709     0.144151  
std       0.271761     0.200096  
min       0.000886     0.000469  
25%       0.012929     0.019475  
50%       0.037109     0.062918  
75%       0.169766     

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.029641,0.338238,0.983973,0.949028,0.697847,0.956065,0.729602,9780002005883
1,0.594470,0.461990,0.935215,0.704422,0.891110,0.051414,0.212222,9780002261982
2,0.041301,0.024568,0.973285,0.767237,0.042176,0.010860,0.009796,9780006178736
3,0.325304,0.125763,0.436339,0.242209,0.732687,0.043272,0.029084,9780006280897
4,0.091733,0.197434,0.095043,0.041146,0.890048,0.475881,0.074878,9780006280934


## 7. Merging Emotion Scores with Book Data

We'll merge the emotion scores back into the main book dataset using the ISBN as the key.


In [22]:
# Merge emotion scores with original book data
df_books_enriched = df_books.merge(emotion_scores_df, on='isbn13', how='inner')

print(f"Enriched dataset size: {len(df_books_enriched)} books")
print(f"Original dataset size: {len(df_books)} books")
print(f"\nNew columns added: {emotion_types}")
print(f"\nTotal columns: {len(df_books_enriched.columns)}")

# Display sample of enriched data
df_books_enriched[['title', 'isbn13'] + emotion_types].head()


Enriched dataset size: 5197 books
Original dataset size: 5197 books

New columns added: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

Total columns: 21


,title,isbn13,anger,disgust,fear,joy,sadness,surprise,neutral
0,Gilead,9780002005883,0.029641,0.338238,0.983973,0.949028,0.697847,0.956065,0.729602
1,Spider's Web,9780002261982,0.594470,0.461990,0.935215,0.704422,0.891110,0.051414,0.212222
2,Rage of angels,9780006178736,0.041301,0.024568,0.973285,0.767237,0.042176,0.010860,0.009796
3,The Four Loves,9780006280897,0.325304,0.125763,0.436339,0.242209,0.732687,0.043272,0.029084
4,The Problem of Pain,9780006280934,0.091733,0.197434,0.095043,0.041146,0.890048,0.475881,0.074878


## 8. Exporting the Enriched Dataset

Finally, we'll save the dataset with emotion scores to a CSV file for use in downstream analysis and recommendation systems.


In [23]:
# Export the enriched dataset
output_filename = 'books_with_emotions.csv'
df_books_enriched.to_csv(output_filename, index=False)

print(f"Dataset exported to: {output_filename}")
print(f"Total books: {len(df_books_enriched)}")
print(f"Emotion features: {emotion_types}")
print(f"\nDataset columns: {list(df_books_enriched.columns)}")


Dataset exported to: books_with_emotions.csv
Total books: 5197
Emotion features: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

Dataset columns: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'full_title', 'indexed_content', 'genre', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
